In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, re, functools, json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType
from src import image_service

pd.set_option('display.max_columns', None)

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

os.getcwd()

In [ ]:
def get_commanders_from_txt_file(fpath: str) -> list[str]:
    with open(fpath, 'r') as f:
        return [x.strip() for x in f.read().split("\n")]

def get_json_file(fpath: str) -> dict:
    with open(fpath, 'r') as f:
        return json.loads(f.read())

In [ ]:
sfds = ScryfallDataService(
    base_url="https://api.scryfall.com/", 
    json_cache_service=CacheService("./cache/json/", cache_type=CacheType.JSON), 
    png_cache_service=CacheService("./cache/png/", cache_type=CacheType.PNG), 
    http_client=ScryfallHttpService(),
)

# sfds.clear_caches()

In [ ]:
cards_df = sfds.get_cards_from_bulk_data('gameplay_columns')
cards_df.shape

In [ ]:
all_keywords_array = sorted(set([x.lower() for x in cards_df.keywords.explode().tolist() if isinstance(x, str)]))

In [ ]:
def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

In [ ]:
def drop_all_na_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_remove = []
    for c in df.columns:
        if df[~df[c].isna()].shape[0] > 0:
            continue 
        cols_to_remove.append(c)
    return df.drop(columns=cols_to_remove)

def rename_bool_cols(df: pd.DataFrame) -> pd.DataFrame:
    fix_col_name = lambda c: re.sub("^", "is_", c) if df[c].dtype == bool and len(re.findall("^is_", c)) == 0 else c
    df.columns = [fix_col_name(c) for c in df.columns]
    return df

In [ ]:
def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

def parse_scryfall_data(df: pd.DataFrame, magic_metadata_dict: dict) -> pd.DataFrame:
    df['is_split_card'] = df['name'].apply(lambda x: "//" in x)
    df['supertypes'] = df['type_line'].apply(get_supertypes_from_type_line)
    df['types'] = df['type_line'].apply(functools.partial(get_types_from_type_line, valid_card_types=magic_metadata_dict.get("type_and_subtype_info").keys()))
    df['subtypes'] = df['type_line'].apply(functools.partial(get_subtypes_from_type_line, type_to_subtype_dict=magic_metadata_dict.get("type_and_subtype_info")))
    df['oracle_text_mod'] = df.oracle_text.apply(lambda x: x.split("\n"))
    return df

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return df[[x for x in columns if x in df.columns]]

commander_df = cards_df[cards_df.is_commander_legal & ~cards_df.oracle_text.isna() & cards_df.keywords.str.len().ge(3)].copy() \
    .pipe(drop_all_na_columns) \
    .pipe(rename_bool_cols) \
    .pipe(functools.partial(parse_scryfall_data, magic_metadata_dict=get_json_file("./data/magic_metadata.json"))) \
    .pipe(functools.partial(select_columns, columns=['oracle_id', 'oracle_text', 'oracle_text_mod']))


In [ ]:
# commander_df: pd.DataFrame = cards_df[
#     cards_df.is_commander_legal & ~cards_df.oracle_text.isna() & 
#     cards_df.keywords.str.len().ge(3)
# ].sample(frac=1).copy()
# commander_df.keywords = commander_df.keywords.apply(lambda arr: [x.lower() for x in arr])
# commander_df.oracle_text = commander_df.oracle_text.apply(lambda x: x.lower())
# commander_df = commander_df[['oracle_text', 'keywords']]

# def get_keywords_regex(keywords: list[str]) -> str:
#     return r"(?:^|, ){1}(" + "|".join(keywords) + ")"

# orr_iterable = zip(commander_df.oracle_text.tolist(), commander_df.keywords.tolist())
# for text, keywords in [x for x in orr_iterable][:10]:
#     print(keywords)

#     abilities = text.split("\n")
#     print(abilities)
#     for a in abilities:
#         matches = re.findall(get_keywords_regex(all_keywords_array), a)
#         print("\t{}".format(matches))

#     print("")

In [ ]:
search_text = "whenever a creature you control attacks"
search_embedding = model.encode(search_text)
commander_df['oracle_text_embedding'] = commander_df.oracle_text.apply(lambda x: model.encode(x))
commander_df['search_similarity'] = commander_df.oracle_text_embedding.apply(lambda x: util.cos_sim(x, search_embedding).item())

for otext in commander_df.sort_values(by='search_similarity', ascending=False).head(10)['oracle_text'].tolist():
    print(f'\n{otext}\n')